# AIE464 — Kaggle Full Training (รออนุมัติ)
ใช้ source/models/loss/manifest และ train_experiment เดิมจาก Input แบบ Read-only
4 โมเดล: CNN + L1, DnCNN-lite, U-Net + L1, U-Net + L1 + 0.1 × Sobel Loss

ค่าเริ่มต้น: 20 epochs เท่ากันทุกโมเดล, seed 42, patch 256, batch 1, Adam lr=0.001,
GPU 0/AMP เลือก best checkpoint ด้วย Validation PSNR และ **ไม่ประเมิน Test Set**

TRAINING_APPROVED=False: Run All จะหยุดก่อน Training ต้องแนบ GPU Smoke Backup ZIP และ checksum
เพื่อตรวจผลจริงและประมาณเวลาฝึกก่อนยืนยัน TIME_BUDGET_CONFIRMED แล้วขออนุมัติ Training
Smoke ผ่านพิสูจน์ pipeline ทำงาน ไม่พิสูจน์ว่าโมเดลมีคุณภาพหรือ converge


In [ ]:
import os, sys
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
sys.dont_write_bytecode = True  # Do not write __pycache__ into read-only Input.
from pathlib import Path
import json, hashlib, zipfile, math, uuid
from datetime import datetime, timezone

PROJECT_DIR = Path('/kaggle/input/datasets/thames98451/aie464-denoising-benchmark/mini_project')
WORKING_DIR = Path('/kaggle/working')
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'Attach the existing benchmark Dataset: {PROJECT_DIR}')
if not PROJECT_DIR.resolve().is_relative_to(Path('/kaggle/input').resolve()):
    raise ValueError('Source and data must be read from Kaggle Input')
if not WORKING_DIR.is_dir():
    raise FileNotFoundError('This notebook must run in Kaggle')

EXPECTED_SOURCE_HASHES = {
    "dataset.py": "ab2481e18f3168b618a6cb3282200336785aba2e737ccb56f99e61ffe8bd8134",
    "evaluate.py": "39014632456859ffcd9810b3663f3647cab2cabd0caa98a6f57b9f52cbb1b41a",
    "benchmark.py": "a88fbf9e71106a3e111bce344bdd4cb0674148ef83134e1ede1fe4c113ee8bf9",
    "train.py": "bd5fc026ed342e523b12a0ed86651b3e19d0a60653a25f4bfff87445704e0de2",
    "losses.py": "cefc6c2bf8e8442c69e972cc9022f59455e10ab84f90fb360cc079bea8c7d1bd",
    "models/shallow_cnn.py": "46d09daa2e6796985395e4874eeacdf632bbfccb548b3c37cd40f5a6bb3b0cd9",
    "models/dncnn.py": "79399e92ea6d87d06dd7cf2868f0afef5bf1acc530f09f0d284b686ae0d13bdb",
    "models/unet.py": "c855089306999fd5eed5f695a5caee706c47677249898aba1db5c2968b6ea2c7"
}
for name, expected in EXPECTED_SOURCE_HASHES.items():
    actual = hashlib.sha256((PROJECT_DIR / 'src' / name).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f'Source differs from the approved pipeline: {name}. Update the Input Dataset; do not edit the pipeline here.')

sys.path.insert(0, str(PROJECT_DIR / 'src'))
import torch, numpy, PIL, matplotlib
from train import TrainConfig, train_experiment, source_hashes, versions
from benchmark import EXPERIMENTS, load_manifest, json_save, MEMORY_LIMIT, MANIFEST_SHA256, verify_selected_checkpoint
# Reject stale imports from another project in a reused kernel.
for name in ('train', 'benchmark', 'dataset', 'evaluate', 'losses',
             'models.shallow_cnn', 'models.dncnn', 'models.unet'):
    if not Path(sys.modules[name].__file__).resolve().is_relative_to((PROJECT_DIR / 'src').resolve()):
        raise RuntimeError('Restart the kernel: source modules were imported from another directory')
manifest = load_manifest(PROJECT_DIR)  # Metadata only, including split separation.
print('Frozen split counts:', manifest['counts'])


## Config และ Approval Gates
ตั้ง SMOKE_ARCHIVE เป็น ZIP ล่าสุดของ Notebook 04 ที่อัปโหลดเป็น Input ส่วนตัว
คัดลอก SHA256 จากไฟล์ `.sha256` เป็น SMOKE_SHA256 ไม่ใช้ checkpoint Smoke มา Resume Training

รัน cell ตรวจ Smoke/เวลาได้โดยไม่ Training จากนั้นอ่านประมาณการและตั้ง TIME_BUDGET_CONFIRMED=True
ตั้ง TRAINING_APPROVED=True **เฉพาะหลังได้รับอนุมัติ** เปลี่ยน RUN_ID สำหรับ Training run ใหม่


In [ ]:
TRAINING_APPROVED = False
TIME_BUDGET_CONFIRMED = False
RUN_FINAL_TEST = False
FINAL_TEST_APPROVED = False
MODE, EPOCHS, SEED, PATCH_SIZE = 'train', 20, 42, 256
RUN_ID = 'kaggle_full_20261013_01'
SMOKE_ARCHIVE = None  # Path('/kaggle/input/<private-smoke-backup>/<snapshot>.zip')
SMOKE_SHA256 = '647250700fe5165bbb46bc42f78a1593722272b0384fd6afab640d2de2b8306a'  # SHA256 from the separately saved .sha256 file.
SESSION_BUDGET_HOURS = None  # Optional actual remaining session/GPU budget, not a Kaggle limit.
RESUME = False
RESTORE_ARCHIVE = None  # FULL TRAINING backup only; never restore a smoke checkpoint here.
EXPECTED_BACKUP_SHA256 = ''


def validate_training_config():
    if RUN_FINAL_TEST or FINAL_TEST_APPROVED:
        raise PermissionError('Test evaluation is prohibited in this notebook')
    if MODE != 'train' or SEED != 42 or PATCH_SIZE != 256:
        raise ValueError('Keep the approved train/seed/patch protocol')
    if not isinstance(EPOCHS, int) or isinstance(EPOCHS, bool) or not 1 <= EPOCHS <= 20:
        raise ValueError('Default budget is 20 epochs; any reduction must apply equally and be approved')
    if Path(RUN_ID).name != RUN_ID or RUN_ID in ('', '.', '..'):
        raise ValueError('RUN_ID must be one folder name')


def validate_time_budget(report):
    if SESSION_BUDGET_HOURS is not None:
        if not math.isfinite(SESSION_BUDGET_HOURS) or SESSION_BUDGET_HOURS <= 0:
            raise ValueError('Enter your actual positive remaining GPU/session budget')
        if report['estimated_hours_high'] > SESSION_BUDGET_HOURS:
            raise RuntimeError('Estimated upper bound exceeds budget; obtain approval for an equal reduced budget or separate sessions')


def require_training_approval():
    validate_training_config()
    if not TRAINING_APPROVED:
        raise PermissionError('Wait for separate Full Training approval')
    if not TIME_BUDGET_CONFIRMED or not globals().get('SMOKE_REVIEWED', False):
        raise PermissionError('Review GPU Smoke evidence and time budget before Training')
    if not globals().get('ENVIRONMENT_VERIFIED', False):
        raise PermissionError('CUDA/AMP/environment preflight must succeed before Training')
    if (estimate['epochs_per_model'] != EPOCHS or estimate['updates_per_model'] != 100 * EPOCHS or
            estimate['source_hashes'] != EXPECTED_SOURCE_HASHES or estimate['smoke_archive_sha256'] != SMOKE_SHA256):
        raise ValueError('Config or smoke input changed after estimation; rerun the review cells')
    validate_time_budget(estimate)
    current = versions(torch.device('cuda'))
    if current != estimate['measured_smoke_versions'] or current != preflight['versions']:
        raise RuntimeError('Current environment no longer matches the verified Smoke/preflight')
    if source_hashes(PROJECT_DIR) != EXPECTED_SOURCE_HASHES:
        raise RuntimeError('Source changed after preflight')


validate_training_config()
RUN_DIR = WORKING_DIR / 'final_benchmark' / RUN_ID
BACKUP_DIR = WORKING_DIR / 'backups'
SMOKE_REVIEWED = False
ENVIRONMENT_VERIFIED = False
preflight = None
print('Training output:', RUN_DIR, '| Approval:', TRAINING_APPROVED)


## ตรวจ Backup Smoke และประมาณเวลา — ไม่มี Training
ตรวจ ZIP + SHA256, source/manifest, CUDA/AMP, finite losses, training ID/crop และ U-Net initialization
อ่าน runtime/memory จริงจากผล Smoke ของทั้ง 4 โมเดล ไม่เปิดภาพ Test

เวลาคาดการณ์จากหนึ่ง update ยังหยาบมาก: epoch ≈ 100 × เวลา Smoke train + 10 × เวลา Val สองหน้า
คูณจำนวน epochs และแสดงช่วง 0.5–2 เท่าของค่าประมาณ พร้อมเผื่อ 20% IO/checkpoint/backup
GPU warm-up, page sizes และโหลดเครื่องทำให้เวลาจริงต่างได้ ค่า memory จาก 1 update ไม่รับประกัน peak
ของ Validation 20 หน้า ต้องใช้ guard 6 GiB ของ pipeline ต่อไป


In [ ]:
def inspect_smoke_archive(archive, checksum):
    if archive is None:
        raise FileNotFoundError('Attach your saved GPU Smoke ZIP and set SMOKE_ARCHIVE/SMOKE_SHA256')
    archive = Path(archive)
    if len(checksum) != 64 or hashlib.sha256(archive.read_bytes()).hexdigest() != checksum:
        raise ValueError('Smoke ZIP checksum differs from your saved .sha256')
    with zipfile.ZipFile(archive) as bundle:
        names = bundle.namelist()
        if len(names) != len(set(names)) or bundle.testzip() is not None:
            raise ValueError('Smoke ZIP integrity failure')
        metadata = json.loads(bundle.read('backup_manifest.json'))
        if metadata['mode'] != 'smoke' or metadata['epochs'] != 1 or metadata['source_hashes'] != EXPECTED_SOURCE_HASHES:
            raise ValueError('Smoke used another source or protocol')
        smoke_id = metadata['run_id']
        if Path(smoke_id).name != smoke_id or smoke_id in ('', '.', '..'):
            raise ValueError('Invalid Smoke run ID')
        records = metadata['files']
        if set(names) != set(records) | {'backup_manifest.json'}:
            raise ValueError('Smoke backup manifest coverage differs')
        for name, record in records.items():
            content = bundle.read(name)
            if len(content) != record['bytes'] or hashlib.sha256(content).hexdigest() != record['sha256']:
                raise ValueError('Smoke backup file checksum differs')
        def read_json(relative):
            return json.loads(bundle.read(f'{smoke_id}/{relative}'))
        verification = read_json('smoke_verification.json')
        if (verification['test_evaluated'] or not verification['same_training_id_crop'] or
                not verification['same_unet_initialization'] or
                not verification['amp_preflight']['grad_scaler_enabled'] or
                verification['amp_preflight']['amp_probe_dtype'] != 'torch.float16'):
            raise ValueError('GPU Smoke verification did not pass')
        estimates, environments, samples = [], [], []
        for experiment in EXPERIMENTS:
            metrics = read_json(f'{experiment}/metrics.json')
            config = metrics['config']
            step_log = read_json(f'{experiment}/step_log.json')
            epochs = read_json(f'{experiment}/validation.json')
            check = verification['checks'][experiment]
            if (metrics['mode'] != 'smoke' or metrics['steps'] != 1 or metrics['test_evaluated'] or
                    metrics['manifest_sha256'] != MANIFEST_SHA256 or config['experiment'] != experiment or
                    config['mode'] != 'smoke' or config['epochs'] != 1 or config['seed'] != 42 or
                    config['patch_size'] != 256 or config['learning_rate'] != 0.001 or config['edge_weight'] != 0.1 or
                    check['optimizer_updates'] != 1 or not check['finite_forward_backward']):
                raise ValueError(f'Invalid smoke evidence: {experiment}')
            if metrics['versions']['device'] != 'cuda' or not metrics['versions']['cuda']:
                raise ValueError('CPU smoke cannot estimate a Kaggle GPU run')
            if len(step_log) != 1 or step_log[0]['id'] not in manifest['splits']['train']:
                raise ValueError('Smoke training coverage differs')
            if not all(math.isfinite(step_log[0][k]) for k in ('total', 'l1', 'edge', 'grad_norm')):
                raise ValueError('Non-finite Smoke forward/backward')
            if (len(epochs) != 1 or len(epochs[0]['rows']) != 2 or
                    any(r['id'] not in manifest['splits']['val'] for r in epochs[0]['rows'])):
                raise ValueError('Smoke validation coverage differs')
            peak = max(metrics['gpu_memory'].values())
            if peak <= 0 or peak > MEMORY_LIMIT:
                raise ValueError('Smoke GPU memory failed the 6 GiB budget')
            train_seconds, val_seconds = epochs[0]['train_seconds'], epochs[0]['seconds']
            if not all(math.isfinite(x) and x > 0 for x in (train_seconds, val_seconds)):
                raise ValueError('Smoke timings are missing or invalid')
            nominal = EPOCHS * (100 * train_seconds + 10 * val_seconds) + 10 * metrics['validation']['seconds']
            nominal *= 1.2  # Planning allowance for IO/checkpoints/ZIP, not a measurement.
            estimates.append({'experiment': experiment, 'smoke_train_seconds': train_seconds,
                              'smoke_val_2_pages_seconds': val_seconds, 'smoke_peak_gpu_gib': peak / 1024**3,
                              'estimated_minutes_low': 0.5 * nominal / 60,
                              'estimated_minutes_high': 2 * nominal / 60})
            environments.append(metrics['versions'])
            samples.append((step_log[0]['id'], tuple(step_log[0]['crop'])))
        if len(set(samples)) != 1 or any(env != environments[0] for env in environments[1:]):
            raise ValueError('Smoke data sampling or environments differ across models')
        return {'smoke_archive_sha256': checksum, 'source_hashes': EXPECTED_SOURCE_HASHES,
                'epochs_per_model': EPOCHS, 'updates_per_model': 100 * EPOCHS,
                'measured_smoke_versions': environments[0], 'models': estimates,
                'estimated_hours_low': sum(r['estimated_minutes_low'] for r in estimates) / 60,
                'estimated_hours_high': sum(r['estimated_minutes_high'] for r in estimates) / 60,
                'estimated_not_measured_training': True, 'test_evaluated': False}


SMOKE_REVIEWED = False
ENVIRONMENT_VERIFIED = False
estimate = inspect_smoke_archive(SMOKE_ARCHIVE, SMOKE_SHA256)
for row in estimate['models']:
    print(row)
print('Estimated total hours (NOT measured Full Training):',
      estimate['estimated_hours_low'], 'to', estimate['estimated_hours_high'])
validate_time_budget(estimate)
SMOKE_REVIEWED = True


## ZIP Backup / Resume
ใช้กลไก ZIP เดิมสำหรับ **Training run เท่านั้น** ไม่ย้าย checkpoint จาก Smoke
Resume ใช้ RUN_ID, EPOCHS/config/source/environment เดิม หากลดงบต้องเริ่ม run ใหม่ทั้ง 4 โมเดล


In [ ]:
ARTIFACT_NAMES = ('best.pt', 'last.pt', 'config.json', 'step_log.json',
                  'validation.json', 'metrics.json', 'loss.png', 'comparisons.png')
RUN_METADATA = ('kaggle_preflight.json', 'training_estimate.json', 'training_verification.json')


def allowed_backup_names():
    return {f'{RUN_ID}/{experiment}/{name}' for experiment in EXPERIMENTS for name in ARTIFACT_NAMES} | {
        f'{RUN_ID}/{name}' for name in RUN_METADATA}


def backup_run():
    records, contents = {}, {}
    for name in sorted(allowed_backup_names()):
        path = RUN_DIR.parent / name
        if path.is_file():
            if path.is_symlink():
                raise ValueError('Do not archive symlinks')
            contents[name] = path.read_bytes()
            records[name] = {'sha256': hashlib.sha256(contents[name]).hexdigest(),
                             'bytes': len(contents[name])}
    if not records:
        return None
    BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    archive = BACKUP_DIR / f'{RUN_ID}_{stamp}_{uuid.uuid4().hex[:8]}.zip'
    metadata = {'run_id': RUN_ID, 'mode': MODE, 'epochs': EPOCHS,
                'source_hashes': EXPECTED_SOURCE_HASHES, 'files': records}
    with zipfile.ZipFile(archive, mode='x', compression=zipfile.ZIP_DEFLATED) as bundle:
        for name in records:
            bundle.writestr(name, contents[name])
        bundle.writestr('backup_manifest.json', json.dumps(metadata, indent=2))
    with zipfile.ZipFile(archive) as bundle:
        if bundle.testzip() is not None:
            raise RuntimeError('Backup ZIP integrity check failed')
    checksum = hashlib.sha256(archive.read_bytes()).hexdigest()
    sidecar = archive.with_suffix('.zip.sha256')
    sidecar.write_text(f'{checksum}  {archive.name}\n')
    print('Download this snapshot + checksum before closing the session:', archive, sidecar)
    return archive


def restore_backup(archive, checksum):
    archive = Path(archive)
    if RUN_DIR.exists():
        raise FileExistsError('Restore never overwrites an existing run')
    if len(checksum) != 64 or hashlib.sha256(archive.read_bytes()).hexdigest() != checksum:
        raise ValueError('Provide the matching external backup SHA256')
    with zipfile.ZipFile(archive) as bundle:
        names = bundle.namelist()
        if len(names) != len(set(names)) or bundle.testzip() is not None:
            raise ValueError('Invalid backup ZIP')
        metadata = json.loads(bundle.read('backup_manifest.json'))
        records = metadata['files']
        if (metadata['run_id'] != RUN_ID or metadata['mode'] != MODE or metadata['epochs'] != EPOCHS
                or metadata['source_hashes'] != EXPECTED_SOURCE_HASHES):
            raise ValueError('Backup protocol differs')
        if not set(records).issubset(allowed_backup_names()) or set(names) != set(records) | {'backup_manifest.json'}:
            raise ValueError('Backup contains unexpected files')
        # Validate every path and content before writing any restored file.
        for name, record in records.items():
            destination = (RUN_DIR.parent / name).resolve()
            content = bundle.read(name)
            if (not destination.is_relative_to(RUN_DIR.resolve()) or len(content) != record['bytes']
                    or hashlib.sha256(content).hexdigest() != record['sha256']):
                raise ValueError('Backup path or contents differ')
        for name in records:
            destination = RUN_DIR.parent / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(bundle.read(name))
    print('Restored to Working; the unchanged pipeline will verify Resume provenance.')


if RESTORE_ARCHIVE is not None:
    if not RESUME:
        raise ValueError('Set RESUME=True to use a restored run')
    restore_backup(RESTORE_ARCHIVE, EXPECTED_BACKUP_SHA256)
if RESUME and not RUN_DIR.is_dir():
    raise FileNotFoundError('Restore your external backup or select an existing run before Resume')
if not RESUME and RUN_DIR.exists():
    raise FileExistsError('Choose a new RUN_ID; never overwrite an existing run')


## CUDA/AMP และ Environment
ตรวจ CUDA/AMP และเทียบ GPU/software กับผล Smoke ก่อนอนุมัติเวลา ไม่เริ่ม optimizer ที่ cell นี้
ถ้า environment ต่างกันให้หยุดและขอประเมินใหม่ ไม่เปลี่ยน pipeline ให้ผ่านเอง


In [ ]:
ENVIRONMENT_VERIFIED = False
preflight = None
validate_training_config()
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator; no CPU fallback')
torch.cuda.set_device(0)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(0)
with torch.inference_mode(), torch.amp.autocast(device_type='cuda', dtype=torch.float16):
    probe = torch.ones(16, 16, device='cuda') @ torch.ones(16, 16, device='cuda')
if probe.dtype != torch.float16 or not torch.isfinite(probe).all():
    raise RuntimeError('CUDA float16 autocast probe failed')
scaler_probe = torch.amp.GradScaler('cuda')
if not scaler_probe.is_enabled():
    raise RuntimeError('CUDA GradScaler unavailable')
torch.cuda.synchronize(0)
preflight = {'device': 'cuda:0', 'gpu': torch.cuda.get_device_name(0),
             'gpu_count': torch.cuda.device_count(), 'torch': str(torch.__version__),
             'cuda': torch.version.cuda, 'amp_probe_dtype': str(probe.dtype),
             'grad_scaler_enabled': scaler_probe.is_enabled(),
             'source_hashes': source_hashes(PROJECT_DIR), 'mode': MODE, 'epochs': EPOCHS,
             'test_evaluated': False}
del probe, scaler_probe
preflight['versions'] = versions(torch.device('cuda'))
if not SMOKE_REVIEWED or preflight['versions'] != estimate['measured_smoke_versions']:
    raise RuntimeError('Current GPU/software differs from Smoke; review timings and rerun approved smoke in this environment')
print('Environment matches the measured GPU Smoke.')
print(preflight)
ENVIRONMENT_VERIFIED = True


## Full Training — ต้องอนุมัติก่อนรัน
เมื่ออ่านประมาณการและได้รับอนุมัติแล้ว ให้ตั้ง TRAINING_APPROVED=True และ TIME_BUDGET_CONFIRMED=True
คงค่าเริ่มต้น 20 epochs ทั้ง 4 โมเดล Pipeline เดิมใช้ Train 100 crops/epoch และ Val 20 หน้าเต็ม/epoch
เลือก checkpoint ด้วย Val PSNR ไม่สร้าง Test Dataset และไม่เรียก Final Test


In [ ]:
require_training_approval()
RUN_DIR.mkdir(parents=True, exist_ok=True)
json_save(estimate, RUN_DIR / 'training_estimate.json')
json_save(preflight, RUN_DIR / 'kaggle_preflight.json')
summaries = {}
for experiment in EXPERIMENTS:
    folder = RUN_DIR / experiment
    config = TrainConfig(experiment=experiment, mode=MODE, epochs=EPOCHS,
                         seed=SEED, patch_size=PATCH_SIZE)
    try:
        summaries[experiment] = train_experiment(
            PROJECT_DIR, folder, config,
            resume=RESUME and folder.exists(),
            approved_training=TRAINING_APPROVED,
            require_cuda=True,
        )
        summary = summaries[experiment]
        if summary['steps'] != 100 * EPOCHS or summary['test_evaluated']:
            raise RuntimeError('Full Training scope or update budget violated')
        print(experiment, '| best epoch:', summary['selected_epoch'],
              '| Validation:', summary['validation']['mean'], '| GPU memory:', summary['gpu_memory'])
    finally:
        backup_run()  # Download/save snapshots outside the session yourself.


## ตรวจผล Training และสำรองครั้งสุดท้าย
ตรวจงบ epochs/updates, Val coverage, best checkpoint, memory และความยุติธรรมของ ID/crop sequence
ใช้ Validation เท่านั้น ผล Training สำเร็จไม่ยืนยันว่า proposed model ดีกว่า baseline


In [ ]:
require_training_approval()
checks, sequence_hashes, initial_unet = {}, [], {}
for experiment in EXPERIMENTS:
    folder = RUN_DIR / experiment
    last = torch.load(folder / 'last.pt', map_location='cpu', weights_only=True)
    best = torch.load(folder / 'best.pt', map_location='cpu', weights_only=True)
    verify_selected_checkpoint(last, best)
    history, val_epochs = last['history'], last['validation']
    if last['completed_epochs'] != EPOCHS or len(history) != 100 * EPOCHS or last['test_evaluated']:
        raise RuntimeError('Training checkpoint coverage differs')
    if len(val_epochs) != EPOCHS or any({r['id'] for r in v['rows']} != set(manifest['splits']['val']) or len(v['rows']) != 20 for v in val_epochs):
        raise RuntimeError('Validation protocol differs')
    if any(r['id'] not in manifest['splits']['train'] or not all(math.isfinite(r[k]) for k in ('total', 'l1', 'edge', 'grad_norm')) for r in history):
        raise RuntimeError('Invalid training evidence')
    sequence = [(r['epoch'], r['id'], r['crop']) for r in history]
    sequence_hashes.append(hashlib.sha256(json.dumps(sequence).encode()).hexdigest())
    if experiment.startswith('unet_'):
        initial_unet[experiment] = last['initial_state_sha256']
    peak = max([max(summaries[experiment]['gpu_memory'].values())] + [max(r['peak_allocated_bytes'], r['peak_reserved_bytes']) for r in history])
    if peak > MEMORY_LIMIT:
        raise RuntimeError('Training exceeded the 6 GiB memory budget')
    for name in ARTIFACT_NAMES:
        if not (folder / name).is_file() or (folder / name).stat().st_size == 0:
            raise RuntimeError('Missing artifact: ' + name)
    checks[experiment] = {'epochs': EPOCHS, 'updates': len(history), 'best_epoch': best['completed_epochs'],
                          'best_validation_psnr': last['best_psnr'], 'peak_gpu_bytes': peak,
                          'committed_train_seconds': sum(v['train_seconds'] for v in val_epochs),
                          'committed_validation_seconds': sum(v['seconds'] for v in val_epochs),
                          'validation': summaries[experiment]['validation']['mean']}
if len(set(sequence_hashes)) != 1 or len(set(initial_unet.values())) != 1:
    raise RuntimeError('Models used different training samples/crops or U-Net initialization')
json_save({'checks': checks, 'same_training_id_crop_sequence': True, 'same_unet_initialization': True,
           'test_evaluated': False, 'quality_claim': False}, RUN_DIR / 'training_verification.json')
latest_backup = backup_run()
print('Training artifacts verified; Test is still prohibited.')


In [ ]:
from IPython.display import display, Image, FileLink
for experiment in EXPERIMENTS:
    print(experiment, '| Best Validation checkpoint: Noisy / Denoised / Clean')
    display(Image(filename=str(RUN_DIR / experiment / 'comparisons.png')))
    display(Image(filename=str(RUN_DIR / experiment / 'loss.png')))
print('Use Kaggle Output / file browser to download the ZIP and .sha256 if links are unavailable.')
display(FileLink(str(latest_backup)))
display(FileLink(str(latest_backup.with_suffix('.zip.sha256'))))


## ก่อนจบ Session
ดาวน์โหลด ZIP ล่าสุดและ `.sha256` จาก `/kaggle/working/backups/` ไปเครื่องหรือ Drive
ตรวจว่าไฟล์เปิดได้ หรือบันทึก Notebook Version พร้อม outputs แล้วตรวจ ZIP/checkpoints ใน Output
ไฟล์ใน Working เพียงอย่างเดียวไม่ใช่ backup นอก Session

Pipeline บันทึก last.pt ทุก epoch; หาก Python exception จะพยายามทำ ZIP ใน finally
หาก kernel/session ถูกตัดทันที ZIP อาจไม่ทันสร้าง กู้ได้ถึง snapshot ที่ดาวน์โหลด/บันทึกไว้นอก Session
เท่านั้น หากหลุดกลางโมเดล อาจต้อง replay ตั้งแต่ snapshot ล่าสุด ห้ามอ้างว่าเก็บทุก update ไว้แล้ว

สำหรับ Resume ใช้ Full Training ZIP, SHA256, RUN_ID และ config เดิมใน Input ส่วนตัว
ขออนุมัติแยกก่อน Test evaluation — Notebook นี้ไม่มีโค้ดประเมิน Test
คู่มือ: `docs/KAGGLE_FULL_TRAINING.md`; [Kaggle Notebooks](https://www.kaggle.com/docs/notebooks)
